# Freddie Mac Multi-Year Origination Validation

## Purpose

This notebook applies one reusable Freddie Mac origination pipeline to the
2015–2019 annual sample files. It validates the current 31-field Release 47
layout, preserves the raw fields, creates documented clean and harmonized
fields, derives validated annual-quarter origination cohorts, profiles annual
data quality, compares annual distributions, and verifies every saved output.

### Why use 2015–2019?

The five consecutive pre-pandemic origination cohorts provide a coherent
development window with two useful properties:

- 2015–2016 provide a baseline before Property Valuation Method disclosure
  begins for 2017 originations.
- The window crosses the 2018 Q2 LTV, CLTV, and Number of Borrowers definition
  changes and includes 2019 as the first full calendar year governed by the
  expanded definitions.
- It captures the decline and completion of HARP through December 2018 while
  keeping the initial reliability study separate from the exceptional
  pandemic-era mortgage environment beginning in 2020.

The files are fixed-size annual samples, so percentages describe these samples
and must not be presented as full-population Freddie Mac estimates.

### Documentation basis

- [Freddie Mac Single-Family Loan-Level Dataset General User Guide](https://www.freddiemac.com/fmac-resources/research/pdf/user_guide.pdf)
- [Release 47 disclosure changes](https://www.freddiemac.com/fmac-resources/research/pdf/disclosure-changes-summary.pdf)
- [Historical disclosure changes](https://capitalmarkets.freddiemac.com/crt/docs/pdfs/historical-disclosure-changes.pdf)
- [Single-Family Loan-Level Dataset](https://www.freddiemac.com/research/datasets/sf-loanlevel-dataset)
- [FHFA confirmation of HARP completion on December 31, 2018](https://www.fhfa.gov/news/news-release/fannie-mae-and-freddie-mac-refinance-volume-decreases-in-fourth-quarter-2018)

In [1]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display


CURRENT_DIRECTORY = Path.cwd()

if CURRENT_DIRECTORY.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIRECTORY.parent
else:
    PROJECT_ROOT = CURRENT_DIRECTORY

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


from src.freddie_cleaning import clean_origination
from src.freddie_cohorts import (
    add_origination_cohorts,
    create_origination_cohort_summary,
    validate_origination_cohorts,
)
from src.freddie_ingestion import load_origination
from src.freddie_pipeline import run_origination_pipeline
from src.freddie_quality import create_origination_quality_report
from src.freddie_summary import create_annual_summary
from src.freddie_validation import validate_origination


print("Notebook directory:", CURRENT_DIRECTORY)
print("Project root:", PROJECT_ROOT)

Notebook directory: e:\capstone_project\mortgage-model-reliability\notebooks
Project root: e:\capstone_project\mortgage-model-reliability


## 1. File Configuration

Define the complete study window once. Every later comparison, validation,
test, save, and read-back loop uses `YEARS`; no analysis cell is tied to a
single annual sample.

In [2]:
YEARS = [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]

RAW_DATA_DIRECTORY = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "Freddie_Mac"
)

ORIGINATION_FILES = {
    year: (
        RAW_DATA_DIRECTORY
        / f"sample_{year}"
        / f"sample_orig_{year}.txt"
    )
    for year in YEARS
}

file_check = pd.DataFrame(
    [
        {
            "year": year,
            "exists": file_path.exists(),
            "path": str(file_path),
        }
        for year, file_path in ORIGINATION_FILES.items()
    ]
)

display(file_check)

missing_files = file_check.loc[
    ~file_check["exists"],
    "path",
].tolist()

if missing_files:
    raise FileNotFoundError(
        "Missing annual origination files: "
        f"{missing_files}"
    )

,year,exists,path
0,2015,True,e:\capstone_project\mortgage-model-reliability...
1,2016,True,e:\capstone_project\mortgage-model-reliability...
2,2017,True,e:\capstone_project\mortgage-model-reliability...
3,2018,True,e:\capstone_project\mortgage-model-reliability...
4,2019,True,e:\capstone_project\mortgage-model-reliability...
5,2020,True,e:\capstone_project\mortgage-model-reliability...
6,2021,True,e:\capstone_project\mortgage-model-reliability...
7,2022,True,e:\capstone_project\mortgage-model-reliability...
8,2023,True,e:\capstone_project\mortgage-model-reliability...


## 2. Load and Structurally Validate Annual Data

Each pipe-delimited file is scanned before pandas ingestion. The scan verifies
that every record has 31 fields and that no raw value exceeds its documented
Release 47 maximum field length. The loaded shape is then reported for every
year.

In [3]:
raw_origination = {}
structure_rows = []

for year in YEARS:
    annual_data = load_origination(
        ORIGINATION_FILES[year]
    )

    raw_origination[year] = annual_data

    structure_rows.append(
        {
            "year": year,
            **annual_data.attrs[
                "structure_validation"
            ],
            "loaded_rows": annual_data.shape[0],
            "loaded_columns": annual_data.shape[1],
        }
    )

structure_validation = pd.DataFrame(
    structure_rows
)

display(structure_validation)

,year,rows_scanned,expected_fields_per_row,field_count_errors,field_length_errors,loaded_rows,loaded_columns
0,2015,50000,31,0,0,50000,31
1,2016,50000,31,0,0,50000,31
2,2017,50000,31,0,0,50000,31
3,2018,50000,31,0,0,50000,31
4,2019,50000,31,0,0,50000,31
5,2020,50000,31,0,0,50000,31
6,2021,50000,31,0,0,50000,31
7,2022,50000,31,0,0,50000,31
8,2023,50000,31,0,0,50000,31


### Structural Validation Findings

- All five annual files were found and loaded successfully.
- Each year contains 50,000 sampled loans and 31 raw Release 47 fields, for
  250,000 records in total.
- No row has an incorrect field count or a raw value exceeding the documented
  maximum length.
- These results establish structural consistency; semantic validity is assessed
  separately by the documentation-aware quality rules below.

## 3. Annual Data-Quality Reports

The quality report separates five different concepts:

1. undocumented categorical codes;
2. documented sentinel frequency;
3. raw null frequency;
4. non-sentinel values outside documented numeric ranges; and
5. time-aware definition and LTV/CLTV rule warnings.

This separation prevents a documented unavailable code from being confused
with an ingestion null or an invalid value.

In [4]:
quality_reports = {}
quality_check_rows = []

for year in YEARS:
    annual_report = create_origination_quality_report(
        data=raw_origination[year],
        year=year,
    )

    quality_reports[year] = annual_report

    quality_check_rows.append(
        {
            "year": year,
            "unexpected_code_rows": len(
                annual_report["unexpected_codes"]
            ),
            "numeric_range_warning_records": int(
                annual_report[
                    "numeric_range_warnings"
                ]["count"].sum()
            ),
            "definition_warning_records": int(
                annual_report[
                    "definition_warnings"
                ]["count"].sum()
            ),
            "ratio_warning_records": len(
                annual_report["ratio_warnings"]
            ),
        }
    )

quality_check_summary = pd.DataFrame(
    quality_check_rows
)

display(quality_check_summary)

,year,unexpected_code_rows,numeric_range_warning_records,definition_warning_records,ratio_warning_records
0,2015,0,0,0,0
1,2016,0,0,0,0
2,2017,0,0,0,0
3,2018,0,0,0,0
4,2019,0,0,0,0
5,2020,0,0,0,0
6,2021,0,0,0,0
7,2022,0,0,0,0
8,2023,0,0,0,0


### 3.1 Documented Sentinel Frequencies

Sentinels are documented unavailable, unknown, or not-applicable codes. They
are monitored in raw data and converted to missing only in the corresponding
clean fields. In particular, VantageScore 4.0 is a Release 47 field; historical
sample rows may legitimately contain `9999` throughout.

In [5]:
sentinel_counts = pd.concat(
    [
        quality_reports[year]["sentinel_counts"]
        for year in YEARS
    ],
    ignore_index=True,
)

nonzero_sentinels = (
    sentinel_counts.loc[
        sentinel_counts["count"] > 0,
        [
            "year",
            "column",
            "sentinel",
            "count",
            "percentage",
        ],
    ]
    .sort_values(
        by=["column", "year"]
    )
    .reset_index(drop=True)
)

display(nonzero_sentinels.round(3))

,year,column,sentinel,count,percentage
0,2017,CLASSIC FICO,9999,8,0.016
1,2018,CLASSIC FICO,9999,15,0.030
2,2019,CLASSIC FICO,9999,18,0.036
3,2020,CLASSIC FICO,9999,7,0.014
4,2021,CLASSIC FICO,9999,7,0.014
5,2022,CLASSIC FICO,9999,8,0.016
6,2023,CLASSIC FICO,9999,17,0.034
7,2015,FIRST TIME HOMEBUYER INDICATOR,9,8,0.016
8,2016,FIRST TIME HOMEBUYER INDICATOR,9,2,0.004
9,2015,ORIGINAL COMBINED LOAN-TO-VALUE (CLTV),999,1,0.002


### 3.2 Property Valuation Method Availability

Usable Property Valuation Method disclosure begins with 2017 originations.
Codes `1`–`3` are usable methods in this study window; code `7` means
unavailable. Code `4` (ACE plus Property Data Report) does not become
effective until 2022 and therefore should not appear in 2015–2019.

In [6]:
property_valuation_rows = []

for year in YEARS:
    valuation_method = raw_origination[
        year
    ]["PROPERTY VALUATION METHOD"]

    for value, count in (
        valuation_method
        .value_counts(dropna=False)
        .sort_index()
        .items()
    ):
        property_valuation_rows.append(
            {
                "year": year,
                "property_valuation_method": value,
                "count": int(count),
                "percentage": (
                    count
                    / len(valuation_method)
                    * 100
                ),
            }
        )

property_valuation_comparison = pd.DataFrame(
    property_valuation_rows
)

display(
    property_valuation_comparison.round(3)
)

,year,property_valuation_method,count,percentage
0,2015,7,50000,100.000
1,2016,7,50000,100.000
2,2017,1,445,0.890
3,2017,7,49555,99.110
4,2018,1,1513,3.026
5,2018,7,48487,96.974
6,2019,1,5436,10.872
7,2019,7,44564,89.128
8,2020,1,19316,38.632
9,2020,2,30684,61.368


### 3.3 Raw Missingness Comparison

Raw null percentages are aligned across all five years. Documented sentinels
are intentionally excluded from this table because they are reported
separately above. Consecutive-year changes are percentage-point differences.

In [7]:
missingness = pd.concat(
    [
        quality_reports[year]["missingness"]
        for year in YEARS
    ],
    ignore_index=True,
)

missingness_comparison = (
    missingness.pivot(
        index="column",
        columns="year",
        values="missing_percentage",
    )
    .reset_index()
)

missingness_comparison.columns.name = None

missingness_comparison = (
    missingness_comparison.rename(
        columns={
            year: f"missing_{year}_pct"
            for year in YEARS
        }
    )
)

for previous_year, current_year in zip(
    YEARS[:-1],
    YEARS[1:],
):
    missingness_comparison[
        (
            f"change_{current_year}"
            f"_minus_{previous_year}_pp"
        )
    ] = (
        missingness_comparison[
            f"missing_{current_year}_pct"
        ]
        - missingness_comparison[
            f"missing_{previous_year}_pct"
        ]
    )

annual_missingness_columns = [
    f"missing_{year}_pct"
    for year in YEARS
]

nonzero_missingness = (
    missingness_comparison.loc[
        missingness_comparison[
            annual_missingness_columns
        ]
        .gt(0)
        .any(axis=1)
    ]
    .sort_values("column")
    .reset_index(drop=True)
)

display(nonzero_missingness.round(3))

,column,missing_2015_pct,missing_2016_pct,missing_2017_pct,missing_2018_pct,missing_2019_pct,missing_2020_pct,missing_2021_pct,missing_2022_pct,missing_2023_pct,change_2016_minus_2015_pp,change_2017_minus_2016_pp,change_2018_minus_2017_pp,change_2019_minus_2018_pp,change_2020_minus_2019_pp,change_2021_minus_2020_pp,change_2022_minus_2021_pp,change_2023_minus_2022_pp
0,METROPOLITAN STATISTICAL AREA (MSA) OR METROPO...,11.208,10.108,10.804,10.340,9.148,8.216,8.432,10.092,10.470,-1.100,0.696,-0.464,-1.192,-0.932,0.216,1.660,0.378
1,PRE-HARP LOAN SEQUENCE NUMBER,91.966,94.950,96.392,98.978,99.940,100.000,100.000,100.000,100.000,2.984,1.442,2.586,0.962,0.060,0.000,0.000,0.000
2,SPECIAL ELIGIBILITY PROGRAM,99.350,97.884,93.022,88.662,90.634,96.012,96.390,93.594,88.264,-1.466,-4.862,-4.360,1.972,5.378,0.378,-2.796,-5.330


### 3.4 Documentation-Aware Warning Details

Numeric-range warnings cover FICO, VantageScore 4.0, mortgage insurance,
Number of Units, DTI, and Number of Borrowers after excluding their documented
sentinels. Definition warnings enforce effective dates for Number of Borrowers
and Property Valuation Method, Pre-HARP linkage consistency, and the documented
DTI-unavailable rule for HARP/Relief Refinance loans. Ratio warnings apply the
quarter-sensitive rules documented below.

In [8]:
for report_name in [
    "unexpected_codes",
    "numeric_range_warnings",
    "definition_warnings",
    "ratio_warnings",
]:
    annual_tables = [
        quality_reports[year][report_name]
        for year in YEARS
        if not quality_reports[year][
            report_name
        ].empty
    ]

    print(report_name)

    if annual_tables:
        display(
            pd.concat(
                annual_tables,
                ignore_index=True,
            )
        )
    else:
        print("No warnings.")
        print()

unexpected_codes
No warnings.

numeric_range_warnings
No warnings.

definition_warnings
No warnings.

ratio_warnings
No warnings.



### Data-Quality and Availability Findings

- No undocumented categorical codes, numeric-range violations,
  definition-timing violations, or ratio-rule warnings were detected in any
  annual sample.
- DTI sentinel `999` usage fell steadily from 8.040% in 2015 to 0.062% in
  2019. This is an availability change, not a numeric DTI value.
- FICO sentinel `9999` was absent in 2015–2016 and remained rare thereafter:
  0.016% in 2017, 0.030% in 2018, and 0.036% in 2019.
- VantageScore 4.0 is unavailable for every observation in all five samples and
  is therefore unusable for this study window.
- Property Valuation Method is unavailable for 100% of the 2015–2016 samples.
  Usable values begin in 2017 and increase from 0.890% in 2017 to 3.026% in
  2018 and 10.872% in 2019. Every usable value in these samples is code `1`,
  which Freddie Mac documents as an ACE appraisal waiver.
- Raw MSA missingness remains close to 9%–11%. Pre-HARP sequence missingness
  rises from 91.966% to 99.940%, consistent with the shrinking HARP/Relief
  population and HARP's completion at the end of 2018.
- Special Eligibility Program becomes more populated through 2018, followed by
  a modest reversal in 2019. This field should be interpreted with its program
  definition rather than treated as ordinary unexplained missingness.
- Raw nulls and documented sentinel codes are reported separately so disclosure
  changes are not mistaken for borrower-population drift.

The absence of warnings means the observed values conform to the implemented
documentation rules; it does not imply that annual distributions are identical.

## 4. Cleaning, Cohort Construction, and Critical Validation

Cleaning and cohort construction are additive. They preserve all 31 raw fields and append:

- nine numeric clean fields;
- five categorical clean fields;
- a three-character postal-code clean field;
- a harmonized Number of Borrowers group; and
- three origination cohort fields derived from the Freddie Mac loan identifier:
  `ORIGINATION YEAR`, `ORIGINATION QUARTER`, and `ORIGINATION COHORT`.

This produces 50 columns. The harmonized borrower field uses `1` and `2+`
because the raw definition changes in 2018 Q2: earlier values distinguish one
borrower from multiple borrowers, while later values report exact counts from
1 to 10.

The cohort fields preserve the annual source year while retaining quarter-level
origination timing for later drift, target, and model-reliability analysis.

The loan-identifier parser accepts only the documented `PYYQnXXXXXXX` structure,
where `P` is `F` for fixed-rate or `A` for adjustable-rate. It requires quarters
`Q1` through `Q4` and confirms that the identifier year agrees with the annual
source file.

Raw postal code `000` is documented as unavailable and is therefore a valid
missing value after cleaning, not an invalid postal representation.

Validation checks row preservation, raw-field preservation, required clean
fields, sentinel removal, loan identifiers, postal representation, and the
borrower-group derivation. Cohort construction additionally rejects malformed
loan identifiers and source-year mismatches.

In [9]:
cleaned_origination = {}
validation_rows = []

for year in YEARS:
    cleaned_data = clean_origination(
        raw_origination[year]
    )

    cleaned_data = add_origination_cohorts(
        origination=cleaned_data,
        expected_year=year,
    )

    validation_report = validate_origination(
        raw=raw_origination[year],
        cleaned=cleaned_data,
        expected_rows=50000,
    )

    cohort_validation_report = (
        validate_origination_cohorts(
            origination=cleaned_data,
            expected_year=year,
        )
    )

    cleaned_origination[year] = cleaned_data

    validation_rows.append(
        {
            "year": year,
            **validation_report,
            **cohort_validation_report,
        }
    )

validation_summary = pd.DataFrame(
    validation_rows
)

display(validation_summary)

,year,rows,raw_columns,cleaned_columns,missing_loan_ids,duplicate_loan_ids,invalid_postal_codes,invalid_borrower_groups,invalid_origination_years,invalid_origination_quarters,invalid_origination_cohorts
0,2015,50000,31,50,0,0,0,0,0,0,0
1,2016,50000,31,50,0,0,0,0,0,0,0
2,2017,50000,31,50,0,0,0,0,0,0,0
3,2018,50000,31,50,0,0,0,0,0,0,0
4,2019,50000,31,50,0,0,0,0,0,0,0
5,2020,50000,31,50,0,0,0,0,0,0,0
6,2021,50000,31,50,0,0,0,0,0,0,0
7,2022,50000,31,50,0,0,0,0,0,0,0
8,2023,50000,31,50,0,0,0,0,0,0,0


### Cleaning, Cohort, and Validation Findings

- Cleaning preserved all 31 raw fields and appended 16 cleaned or harmonized
  fields plus three derived cohort fields, producing 50 columns for every year.
- Each cleaned dataset retains all 50,000 annual records.
- No missing or duplicate loan identifiers, invalid cleaned postal codes, or
  invalid borrower-group values were detected.
- All loan identifiers followed the documented `PYYQnXXXXXXX` structure.
- Every extracted origination year agreed with its corresponding 2015–2019
  annual source file.
- The new cohort fields retain both annual and quarterly origination timing
  without changing the raw `LOAN IDENTIFIER`.
- Documented unavailable codes are retained in the raw fields for auditability
  and converted to missing only in their corresponding `_CLEAN` fields.
- `NUMBER OF BORROWERS GROUP_CLEAN` provides a comparable `1` versus `2+`
  measure across the 2018 Q2 definition change, when the raw field began
  disclosing exact counts from 1 through 10.

### 4.1 Quarterly Origination Cohort Verification

The Freddie Mac loan identifier provides the origination year and quarter.
The table below verifies the quarterly cohort distribution within each annual
sample.

The annual files remain the reproducible source datasets. Quarter metadata is
retained so later drift and model-reliability analysis can compare quarterly,
semiannual, or annual evaluation windows. The final cadence will depend on
whether each period contains enough target events for reliable measurement.

In [10]:
cohort_summary = pd.concat(
    [
        create_origination_cohort_summary(
            cleaned_origination[year]
        )
        for year in YEARS
    ],
    ignore_index=True,
)

cohort_validation = (
    cohort_summary
    .groupby("year")
    .agg(
        quarter_count=("quarter", "nunique"),
        total_records=("count", "sum"),
        total_percentage=("percentage", "sum"),
    )
    .reset_index()
)

display(cohort_summary.round(3))
display(cohort_validation.round(3))

assert cohort_validation[
    "quarter_count"
].eq(4).all(), (
    "Each annual sample must contain Q1 through Q4."
)

assert cohort_validation[
    "total_records"
].eq(50000).all(), (
    "Quarterly cohort counts do not sum to 50,000."
)

assert cohort_validation[
    "total_percentage"
].sub(100).abs().lt(1e-9).all(), (
    "Quarterly cohort percentages do not sum to 100%."
)

,year,quarter,cohort,count,percentage
0,2015,Q1,2015Q1,12500,25.0
1,2015,Q2,2015Q2,12500,25.0
2,2015,Q3,2015Q3,12500,25.0
3,2015,Q4,2015Q4,12500,25.0
4,2016,Q1,2016Q1,12500,25.0
5,2016,Q2,2016Q2,12500,25.0
6,2016,Q3,2016Q3,12500,25.0
7,2016,Q4,2016Q4,12500,25.0
8,2017,Q1,2017Q1,12500,25.0
9,2017,Q2,2017Q2,12500,25.0


,year,quarter_count,total_records,total_percentage
0,2015,4,50000,100.0
1,2016,4,50000,100.0
2,2017,4,50000,100.0
3,2018,4,50000,100.0
4,2019,4,50000,100.0
5,2020,4,50000,100.0
6,2021,4,50000,100.0
7,2022,4,50000,100.0
8,2023,4,50000,100.0


### Quarterly Cohort Findings

- All 250,000 loans were assigned to one of 20 quarterly origination cohorts
  spanning `2015Q1` through `2019Q4`.
- Every annual sample contains all four expected quarters.
- Each quarterly cohort contains exactly 12,500 loans and represents 25% of its
  annual sample.
- Quarterly counts sum to 50,000 loans and percentages sum to 100% for every
  year.
- No malformed loan identifiers or source-year mismatches were detected.
- The exact quarterly balance is a characteristic of these Freddie Mac sample
  files and must not be interpreted as evidence that actual Freddie Mac
  origination volume was equal across quarters.
- The quarterly cohorts can support comparisons of borrower characteristics,
  outcome rates, drift, and model reliability, but they cannot be used to
  estimate Freddie Mac's quarterly market volume.
- The final analytical cadence—quarterly, semiannual, or annual—will be selected
  after target construction based on the number of observed target events in
  each period.

Freddie Mac documents the sample dataset as a simple random sample of 50,000
loans selected from each full vintage year. The sample supports loan-level
research but should not be presented as the complete Freddie Mac origination
population.

Documentation reference:

- [Freddie Mac Single-Family Loan-Level Dataset General User Guide](https://www.freddiemac.com/fmac-resources/research/pdf/general_user_guide_july_2026.pdf)

## 5. Standardized Annual Summaries

Numeric summaries use cleaned FICO, DTI, LTV, CLTV, mortgage-insurance, and
VantageScore fields so sentinels cannot distort descriptive statistics.
Categorical summaries use cleaned categories and the harmonized borrower
group. These tables are descriptive screening tools; later drift analysis
should apply formal stability tests.

In [11]:
annual_summaries = {}

for year in YEARS:
    annual_summaries[year] = (
        create_annual_summary(
            data=cleaned_origination[year],
            year=year,
        )
    )

numeric_summary = pd.concat(
    [
        annual_summaries[year]["numeric"]
        for year in YEARS
    ],
    ignore_index=True,
)

categorical_summary = pd.concat(
    [
        annual_summaries[year]["categorical"]
        for year in YEARS
    ],
    ignore_index=True,
)

display(numeric_summary.round(3))

,year,variable,count,missing,missing_percentage,mean,std,min,q1,median,q3,max
0,2015,FICO,50000.0,0,0.000,749.759,47.508,462.000,718.00,760.00,788.00,834.000
1,2015,DTI,45980.0,4020,8.040,33.839,9.392,1.000,27.00,35.00,42.00,50.000
2,2015,LTV,49999.0,1,0.002,73.510,17.514,5.000,65.00,78.00,83.00,241.000
3,2015,CLTV,49999.0,1,0.002,74.402,18.021,5.000,66.00,79.00,85.00,854.000
4,2015,UPB,50000.0,0,0.000,224006.060,119990.498,10000.000,131000.00,200000.00,299000.00,974000.000
...,...,...,...,...,...,...,...,...,...,...,...,...
76,2023,UPB,50000.0,0,0.000,311174.640,172692.022,19000.000,182000.00,275000.00,405000.00,1781000.000
77,2023,INTEREST RATE,50000.0,0,0.000,6.736,0.703,2.625,6.25,6.75,7.25,9.125
78,2023,LOAN TERM,50000.0,0,0.000,349.171,42.064,96.000,360.00,360.00,360.00,360.000
79,2023,MI PERCENTAGE,50000.0,0,0.000,10.151,13.344,0.000,0.00,0.00,25.00,35.000


### 5.1 Fixed-Threshold Ratio Comparison

The `>105%` table is retained as a consistent descriptive trend measure
across years. It is **not** a universal validity rule. Freddie Mac's documented
origination rules are:

| Period / population | LTV range | CLTV range |
|---|---:|---:|
| Non-HARP, 2018 Q1 and earlier | 6–105 | 6–200 |
| HARP, 2018 Q1 and earlier | 1–998 | 1–998 |
| 2018 Q2 and later | 1–998 | 1–998 |

Code `999` means unavailable. Freddie Mac also directs that CLTV be reported
as `999` if calculated CLTV is lower than LTV. The separate ratio-warning
report applies these rules at record level.

In [12]:
high_ratio_summary_rows = []
extreme_ratio_records_by_year = {}

for year in YEARS:
    data = cleaned_origination[year]

    ltv = data[
        "ORIGINAL LOAN-TO-VALUE (LTV)_CLEAN"
    ]

    cltv = data[
        "ORIGINAL COMBINED LOAN-TO-VALUE (CLTV)_CLEAN"
    ]

    high_ratio_mask = (
        ltv.gt(105)
        | cltv.gt(105)
    )

    high_ratio_records = data.loc[
        high_ratio_mask
    ].copy()

    extreme_ratio_records_by_year[
        year
    ] = high_ratio_records

    pre_harp_sequence = (
        high_ratio_records[
            "PRE-HARP LOAN SEQUENCE NUMBER"
        ]
        .astype("string")
        .str.strip()
    )

    pre_harp_present = (
        pre_harp_sequence.notna()
        & pre_harp_sequence.ne("")
    )

    high_ratio_summary_rows.append(
        {
            "year": year,
            "records_above_105": len(
                high_ratio_records
            ),
            "percentage_above_105": (
                len(high_ratio_records)
                / len(data)
                * 100
            ),
            "ltv_above_105": int(
                ltv.gt(105).sum()
            ),
            "cltv_above_105": int(
                cltv.gt(105).sum()
            ),
            "harp_y": int(
                high_ratio_records[
                    "HARP INDICATOR"
                ].eq("Y").sum()
            ),
            "pre_harp_id_present": int(
                pre_harp_present.sum()
            ),
            "maximum_ltv": ltv.max(),
            "maximum_cltv": cltv.max(),
        }
    )

high_ratio_summary = pd.DataFrame(
    high_ratio_summary_rows
)

display(high_ratio_summary.round(3))

,year,records_above_105,percentage_above_105,ltv_above_105,cltv_above_105,harp_y,pre_harp_id_present,maximum_ltv,maximum_cltv
0,2015,530,1.060,361,530,530,530,241.0,854.0
1,2016,227,0.454,148,227,227,227,336.0,336.0
2,2017,124,0.248,90,124,124,124,194.0,194.0
3,2018,34,0.068,24,34,34,34,222.0,255.0
4,2019,0,0.000,0,0,0,0,103.0,105.0
5,2020,0,0.000,0,0,0,0,97.0,105.0
6,2021,0,0.000,0,0,0,0,97.0,105.0
7,2022,0,0.000,0,0,0,0,97.0,105.0
8,2023,0,0.000,0,0,0,0,105.0,105.0


### 5.2 Record-Level Review of High-LTV Loans

Display the ten records with the highest cleaned LTV in each year, using CLTV
as the secondary sort. The annual summary above independently reports both the
maximum LTV and maximum CLTV, while the ratio-warning report evaluates every
record against the documentation-aware rules. Empty output is expected for a
year with no value above the fixed 105% descriptive threshold.

In [13]:
extreme_record_columns = [
    "LOAN IDENTIFIER",
    "ORIGINAL LOAN-TO-VALUE (LTV)_CLEAN",
    "ORIGINAL COMBINED LOAN-TO-VALUE (CLTV)_CLEAN",
    "LOAN PURPOSE",
    "HARP INDICATOR",
    "PRE-HARP LOAN SEQUENCE NUMBER",
    "SPECIAL ELIGIBILITY PROGRAM",
    "PROPERTY VALUATION METHOD",
    "PROPERTY VALUATION METHOD_CLEAN",
    "ORIGINAL UPB",
    "PROPERTY TYPE",
    "OCCUPANCY STATUS",
]

for year in YEARS:
    extreme_records = (
        extreme_ratio_records_by_year[year]
        .loc[:, extreme_record_columns]
        .sort_values(
            by=[
                "ORIGINAL LOAN-TO-VALUE (LTV)_CLEAN",
                "ORIGINAL COMBINED LOAN-TO-VALUE (CLTV)_CLEAN",
            ],
            ascending=False,
        )
        .head(10)
        .reset_index(drop=True)
    )

    print(f"Year: {year}")
    display(extreme_records)

Year: 2015


,LOAN IDENTIFIER,ORIGINAL LOAN-TO-VALUE (LTV)_CLEAN,ORIGINAL COMBINED LOAN-TO-VALUE (CLTV)_CLEAN,LOAN PURPOSE,HARP INDICATOR,PRE-HARP LOAN SEQUENCE NUMBER,SPECIAL ELIGIBILITY PROGRAM,PROPERTY VALUATION METHOD,PROPERTY VALUATION METHOD_CLEAN,ORIGINAL UPB,PROPERTY TYPE,OCCUPANCY STATUS
0,F15Q40236260,241.0,241.0,N,Y,F07Q20533709,<NA>,7,NaN,96000,CO,S
1,F15Q10048577,209.0,209.0,N,Y,F05Q20160419,<NA>,7,NaN,104000,SF,P
2,F15Q20301856,200.0,200.0,N,Y,F05Q30048668,<NA>,7,NaN,124000,SF,P
3,F15Q10147868,197.0,197.0,N,Y,F07Q30174420,<NA>,7,NaN,296000,CO,I
4,F15Q20051027,183.0,183.0,N,Y,F05Q30226665,<NA>,7,NaN,132000,SF,P
5,F15Q20078549,182.0,182.0,N,Y,F02Q30116939,<NA>,7,NaN,100000,SF,P
6,F15Q20049593,181.0,181.0,N,Y,A06Q40003040,<NA>,7,NaN,136000,SF,I
7,F15Q40291769,176.0,176.0,N,Y,F07Q10283930,<NA>,7,NaN,133000,CO,P
8,F15Q10307545,175.0,175.0,N,Y,F06Q40325746,<NA>,7,NaN,101000,CO,S
9,F15Q20382069,175.0,175.0,N,Y,F07Q10023410,<NA>,7,NaN,104000,SF,P


Year: 2016


,LOAN IDENTIFIER,ORIGINAL LOAN-TO-VALUE (LTV)_CLEAN,ORIGINAL COMBINED LOAN-TO-VALUE (CLTV)_CLEAN,LOAN PURPOSE,HARP INDICATOR,PRE-HARP LOAN SEQUENCE NUMBER,SPECIAL ELIGIBILITY PROGRAM,PROPERTY VALUATION METHOD,PROPERTY VALUATION METHOD_CLEAN,ORIGINAL UPB,PROPERTY TYPE,OCCUPANCY STATUS
0,F16Q30476204,336,336,N,Y,A08Q30002418,<NA>,7,NaN,138000,CO,P
1,F16Q10200259,260,333,N,Y,F07Q10243093,<NA>,7,NaN,125000,CO,I
2,F16Q20409731,222,222,N,Y,F09Q20311057,<NA>,7,NaN,334000,SF,P
3,F16Q10275314,183,183,N,Y,F06Q40393669,<NA>,7,NaN,155000,SF,P
4,F16Q20236155,175,186,N,Y,F06Q40317266,<NA>,7,NaN,76000,SF,P
5,F16Q10089992,161,161,N,Y,F06Q30152221,<NA>,7,NaN,67000,SF,P
6,F16Q10291861,155,170,N,Y,F07Q30152312,<NA>,7,NaN,212000,SF,P
7,F16Q10226493,155,155,N,Y,F07Q20027043,<NA>,7,NaN,294000,SF,P
8,F16Q10293693,155,155,N,Y,F07Q20020434,<NA>,7,NaN,96000,SF,P
9,F16Q20369946,153,153,N,Y,F08Q10465752,<NA>,7,NaN,123000,CO,P


Year: 2017


...

,LOAN IDENTIFIER,ORIGINAL LOAN-TO-VALUE (LTV)_CLEAN,ORIGINAL COMBINED LOAN-TO-VALUE (CLTV)_CLEAN,LOAN PURPOSE,HARP INDICATOR,PRE-HARP LOAN SEQUENCE NUMBER,SPECIAL ELIGIBILITY PROGRAM,PROPERTY VALUATION METHOD,PROPERTY VALUATION METHOD_CLEAN,ORIGINAL UPB,PROPERTY TYPE,OCCUPANCY STATUS


Year: 2022


,LOAN IDENTIFIER,ORIGINAL LOAN-TO-VALUE (LTV)_CLEAN,ORIGINAL COMBINED LOAN-TO-VALUE (CLTV)_CLEAN,LOAN PURPOSE,HARP INDICATOR,PRE-HARP LOAN SEQUENCE NUMBER,SPECIAL ELIGIBILITY PROGRAM,PROPERTY VALUATION METHOD,PROPERTY VALUATION METHOD_CLEAN,ORIGINAL UPB,PROPERTY TYPE,OCCUPANCY STATUS


Year: 2023


,LOAN IDENTIFIER,ORIGINAL LOAN-TO-VALUE (LTV)_CLEAN,ORIGINAL COMBINED LOAN-TO-VALUE (CLTV)_CLEAN,LOAN PURPOSE,HARP INDICATOR,PRE-HARP LOAN SEQUENCE NUMBER,SPECIAL ELIGIBILITY PROGRAM,PROPERTY VALUATION METHOD,PROPERTY VALUATION METHOD_CLEAN,ORIGINAL UPB,PROPERTY TYPE,OCCUPANCY STATUS


### 5.3 Actual Ratio-Rule Warnings

These are documentation-aware exceptions, not simply all loans above 105%.
An empty table means the observed ratios comply with the applicable
year-quarter, HARP, sentinel, and CLTV-versus-LTV rules.

In [14]:
ratio_warning_tables = [
    quality_reports[year]["ratio_warnings"]
    for year in YEARS
]

ratio_warnings = pd.concat(
    ratio_warning_tables,
    ignore_index=True,
)

print(
    "Total ratio-rule warning records:",
    len(ratio_warnings),
)

display(ratio_warnings)

Total ratio-rule warning records: 0


,year,origination_quarter,LOAN IDENTIFIER,ORIGINAL LOAN-TO-VALUE (LTV),ORIGINAL COMBINED LOAN-TO-VALUE (CLTV),HARP INDICATOR,PRE-HARP LOAN SEQUENCE NUMBER,LOW LTV FLAG,HIGH LTV FLAG,LOW CLTV FLAG,HIGH CLTV FLAG,CLTV BELOW LTV FLAG


### Numeric and Ratio Findings

- Median FICO remains within 752–760 across the five samples; documented
  sentinel values are excluded from the clean summary.
- Median DTI moves from 35 in 2015–2016 to 36 in 2017 and 37 in 2018–2019,
  while DTI unavailability declines substantially.
- Median LTV and CLTV are stable at approximately 78–80. Their extreme annual
  maxima belong to a small, program-supported refinance segment rather than to
  unsupported data errors.
- Median Original UPB is $200,000 in 2015, $215,000 in 2016, $207,000 in
  2017–2018, and $232,000 in 2019.
- Mean interest rate falls from 3.975% in 2015 to 3.791% in 2016, rises to
  4.736% in 2018, and declines to 4.238% in 2019. This is a meaningful annual
  distribution change for later formal drift analysis, not a validation
  failure.
- The fixed-threshold population with LTV or CLTV above 105% contracts from
  530 loans (1.060%) in 2015 to 227 (0.454%) in 2016, 124 (0.248%) in 2017,
  34 (0.068%) in 2018, and zero in 2019.
- Every fixed-threshold high-ratio loan observed from 2015 through 2018 has both
  `HARP INDICATOR = Y` and a populated Pre-HARP Loan Sequence Number.
- The documentation-aware ratio report produces zero warnings on the actual
  data. It applies the pre-change limits before 2018 Q2, the expanded limits
  from 2018 Q2 onward, HARP/Relief exceptions, absolute 1–998 bounds, and the
  rule that CLTV below LTV must be reported as unavailable.
- The synthetic test passes for every year and confirms that the same numeric
  value can be valid or invalid depending on origination quarter and program
  support.

The fixed 105% comparison is retained only as a stable descriptive measure. It
must not replace the date- and program-aware validation rules.

### 5.4 Synthetic Ratio-Rule Test

A synthetic record is a temporary copy whose fields are deliberately changed
to test a rule. It is never written into the source sample or cleaned output.
Each test case has its own identifier and expected result, which is more
reliable than checking only a total warning count.

In [15]:
quality_rule_test_rows = []
quality_rule_warning_tables = []

for year in YEARS:
    year_suffix = str(year)[-2:]

    test_cases = [
        {
            "case": "q1_non_harp_120",
            "quarter": 1,
            "ltv": 120,
            "cltv": 120,
            "expected_warning": year <= 2018,
        },
        {
            "case": "q2_non_harp_120",
            "quarter": 2,
            "ltv": 120,
            "cltv": 120,
            "expected_warning": year < 2018,
        },
        {
            "case": "above_absolute_maximum",
            "quarter": 2,
            "ltv": 1000,
            "cltv": 1000,
            "expected_warning": True,
        },
        {
            "case": "below_minimum",
            "quarter": 2,
            "ltv": 0,
            "cltv": 0,
            "expected_warning": True,
        },
        {
            "case": "cltv_below_ltv",
            "quarter": 2,
            "ltv": 80,
            "cltv": 70,
            "expected_warning": True,
        },
    ]

    quality_rule_test = (
        raw_origination[year]
        .iloc[:len(test_cases)]
        .copy()
    )

    test_identifiers = []

    for position, test_case in enumerate(
        test_cases,
        start=1,
    ):
        test_identifiers.append(
            (
                f"T{year_suffix}Q"
                f"{test_case['quarter']}"
                f"{position:07d}"
            )
        )

    quality_rule_test[
        "LOAN IDENTIFIER"
    ] = test_identifiers

    quality_rule_test[
        "ORIGINAL LOAN-TO-VALUE (LTV)"
    ] = [
        test_case["ltv"]
        for test_case in test_cases
    ]

    quality_rule_test[
        "ORIGINAL COMBINED LOAN-TO-VALUE (CLTV)"
    ] = [
        test_case["cltv"]
        for test_case in test_cases
    ]

    quality_rule_test["HARP INDICATOR"] = "N"

    quality_rule_test[
        "PRE-HARP LOAN SEQUENCE NUMBER"
    ] = pd.NA

    test_report = (
        create_origination_quality_report(
            data=quality_rule_test,
            year=year,
        )
    )

    detected_warnings = test_report[
        "ratio_warnings"
    ]

    detected_identifiers = set(
        detected_warnings[
            "LOAN IDENTIFIER"
        ].astype("string")
    )

    for test_case, identifier in zip(
        test_cases,
        test_identifiers,
    ):
        detected_warning = (
            identifier in detected_identifiers
        )

        quality_rule_test_rows.append(
            {
                "year": year,
                "case": test_case["case"],
                "expected_warning": (
                    test_case[
                        "expected_warning"
                    ]
                ),
                "detected_warning": (
                    detected_warning
                ),
                "test_passed": (
                    detected_warning
                    == test_case[
                        "expected_warning"
                    ]
                ),
            }
        )

    quality_rule_warning_tables.append(
        detected_warnings
    )

quality_rule_test_results = pd.DataFrame(
    quality_rule_test_rows
)

quality_rule_test_summary = (
    quality_rule_test_results
    .groupby("year", as_index=False)
    .agg(
        expected_warnings=(
            "expected_warning",
            "sum",
        ),
        detected_warnings=(
            "detected_warning",
            "sum",
        ),
        test_passed=(
            "test_passed",
            "all",
        ),
    )
)

quality_rule_test_warnings = pd.concat(
    quality_rule_warning_tables,
    ignore_index=True,
)

display(quality_rule_test_summary)
display(quality_rule_test_warnings)

failed_quality_rule_tests = (
    quality_rule_test_results.loc[
        ~quality_rule_test_results[
            "test_passed"
        ]
    ]
)

assert failed_quality_rule_tests.empty, (
    "Ratio-rule tests failed:\n"
    f"{failed_quality_rule_tests}"
)

,year,expected_warnings,detected_warnings,test_passed
0,2015,5,5,True
1,2016,5,5,True
2,2017,5,5,True
3,2018,4,4,True
4,2019,3,3,True
5,2020,3,3,True
6,2021,3,3,True
7,2022,3,3,True
8,2023,3,3,True


,year,origination_quarter,LOAN IDENTIFIER,ORIGINAL LOAN-TO-VALUE (LTV),ORIGINAL COMBINED LOAN-TO-VALUE (CLTV),HARP INDICATOR,PRE-HARP LOAN SEQUENCE NUMBER,LOW LTV FLAG,HIGH LTV FLAG,LOW CLTV FLAG,HIGH CLTV FLAG,CLTV BELOW LTV FLAG
0,2015,1,T15Q10000001,120,120,N,<NA>,False,True,False,False,False
1,2015,2,T15Q20000002,120,120,N,<NA>,False,True,False,False,False
2,2015,2,T15Q20000003,1000,1000,N,<NA>,False,True,False,True,False
3,2015,2,T15Q20000004,0,0,N,<NA>,True,False,True,False,False
4,2015,2,T15Q20000005,80,70,N,<NA>,False,False,False,False,True
5,2016,1,T16Q10000001,120,120,N,<NA>,False,True,False,False,False
6,2016,2,T16Q20000002,120,120,N,<NA>,False,True,False,False,False
7,2016,2,T16Q20000003,1000,1000,N,<NA>,False,True,False,True,False
8,2016,2,T16Q20000004,0,0,N,<NA>,True,False,True,False,False
9,2016,2,T16Q20000005,80,70,N,<NA>,False,False,False,False,True


## 6. Categorical Distribution Comparison

Category percentages are aligned across 2015–2019. Each annual percentage has
an explicit year in its column name, followed by consecutive-year
percentage-point changes. Number of Borrowers uses the comparable `1` versus
`2+` grouping rather than mixing the pre- and post-2018 Q2 raw definitions.

In [16]:
categorical_comparison = (
    categorical_summary.pivot_table(
        index=[
            "variable",
            "category",
        ],
        columns="year",
        values="percentage",
        fill_value=0,
    )
    .reset_index()
)

categorical_comparison.columns.name = None

categorical_comparison = (
    categorical_comparison.rename(
        columns={
            year: f"percentage_{year}"
            for year in YEARS
        }
    )
)

categorical_change_columns = []

for previous_year, current_year in zip(
    YEARS[:-1],
    YEARS[1:],
):
    change_column = (
        f"change_{current_year}"
        f"_minus_{previous_year}_pp"
    )

    categorical_comparison[
        change_column
    ] = (
        categorical_comparison[
            f"percentage_{current_year}"
        ]
        - categorical_comparison[
            f"percentage_{previous_year}"
        ]
    )

    categorical_change_columns.append(
        change_column
    )

categorical_comparison[
    "maximum_absolute_change_pp"
] = (
    categorical_comparison[
        categorical_change_columns
    ]
    .abs()
    .max(axis=1)
)

largest_categorical_changes = (
    categorical_comparison
    .sort_values(
        "maximum_absolute_change_pp",
        ascending=False,
    )
    .head(20)
    .reset_index(drop=True)
)

display(
    largest_categorical_changes.round(3)
)

,variable,category,percentage_2015,percentage_2016,percentage_2017,percentage_2018,percentage_2019,percentage_2020,percentage_2021,percentage_2022,percentage_2023,change_2016_minus_2015_pp,change_2017_minus_2016_pp,change_2018_minus_2017_pp,change_2019_minus_2018_pp,change_2020_minus_2019_pp,change_2021_minus_2020_pp,change_2022_minus_2021_pp,change_2023_minus_2022_pp,maximum_absolute_change_pp
0,LOAN PURPOSE,P,44.308,43.944,57.064,66.826,55.540,30.138,34.276,66.648,84.000,-0.364,13.120,9.762,-11.286,-25.402,4.138,32.372,17.352,32.372
1,LOAN PURPOSE,N,35.026,33.478,20.140,12.298,23.322,50.474,38.190,8.622,3.900,-1.548,-13.338,-7.842,11.024,27.152,-12.284,-29.568,-4.722,29.568
2,FIRST TIME HOMEBUYER,N,84.404,83.908,78.528,73.310,77.490,87.826,86.042,70.704,60.970,-0.496,-5.380,-5.218,4.180,10.336,-1.784,-15.338,-9.734,15.338
3,FIRST TIME HOMEBUYER,Y,15.580,16.088,21.472,26.690,22.510,12.174,13.958,29.296,39.030,0.508,5.384,5.218,-4.180,-10.336,1.784,15.338,9.734,15.338
4,LOAN PURPOSE,C,20.666,22.578,22.796,20.876,21.138,19.388,27.534,24.730,12.100,1.912,0.218,-1.920,0.262,-1.750,8.146,-2.804,-12.630,12.630
5,CHANNEL,R,60.860,60.894,59.346,56.676,52.172,57.696,58.028,53.592,46.506,0.034,-1.548,-2.670,-4.504,5.524,0.332,-4.436,-7.086,7.086
6,CHANNEL,C,29.232,29.308,31.240,33.148,35.580,29.028,28.134,31.416,37.554,0.076,1.932,1.908,2.432,-6.552,-0.894,3.282,6.138,6.552
7,PROPERTY STATE,CA,15.576,15.984,12.956,10.036,11.798,15.620,12.868,8.050,6.924,0.408,-3.028,-2.920,1.762,3.822,-2.752,-4.818,-1.126,4.818
8,NUMBER OF BORROWERS GROUP,2+,51.080,49.436,47.476,45.878,46.126,49.146,46.096,43.610,44.738,-1.644,-1.960,-1.598,0.248,3.020,-3.050,-2.486,1.128,3.050
9,NUMBER OF BORROWERS GROUP,1,48.920,50.564,52.524,54.122,53.874,50.854,53.904,56.390,55.262,1.644,1.960,1.598,-0.248,-3.020,3.050,2.486,-1.128,3.050


### Categorical Distribution Findings

- The largest changes are concentrated in loan purpose. Purchase loans rise
  from 44.308% in 2015 to 66.826% in 2018, then fall to 55.540% in 2019.
  No-cash-out refinance loans follow the inverse pattern, declining from
  35.026% to 12.298% before rebounding to 23.322%.
- First-time homebuyers increase from 15.580% in 2015 to 26.690% in 2018,
  followed by a decrease to 22.510% in 2019.
- Retail channel share declines to 52.172% by 2019, while correspondent and
  broker shares increase to 35.580% and 12.248%, respectively.
- The harmonized single-borrower share rises from 48.920% in 2015 to 54.122%
  in 2018 and is 53.874% in 2019.
- California's sample share declines through 2018 and partially recovers in
  2019; smaller state and property-type changes are also visible.

These percentage-point movements identify candidate product-mix and population
changes. Because the files are annual samples, they are descriptive signals for
formal drift testing and are not full-population estimates or evidence of model
degradation by themselves.

## 7. Save Verified 2015–2019 Outputs

Run the end-to-end pipeline for every configured year. Each annual run saves
the cleaned Parquet data, numeric and categorical summaries, sentinel and
missingness reports, categorical-code warnings, numeric-range warnings,
time-aware definition warnings, ratio warnings, and a validation JSON file.

In [17]:
PROCESSED_DATA_DIRECTORY = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "Freddie_Mac"
)

pipeline_results = {}

for year in YEARS:
    pipeline_result = run_origination_pipeline(
        year=year,
        input_file=ORIGINATION_FILES[year],
        output_directory=(
            PROCESSED_DATA_DIRECTORY
        ),
        expected_rows=50000,
    )

    pipeline_results[year] = pipeline_result

    print(f"Year: {year}")
    print(
        "Structure validation:",
        pipeline_result[
            "structure_validation"
        ],
    )
    print(
        "Data validation:",
        pipeline_result["validation"],
    )
    print(
        "Cohort validation:",
        pipeline_result[
            "cohort_validation"
        ],
    )
    print("Saved outputs:")

    for output_name, output_path in (
        pipeline_result[
            "output_paths"
        ].items()
    ):
        print(
            f"{output_name}: {output_path}"
        )

    print("=" * 70)

Year: 2015
Structure validation: {'rows_scanned': 50000, 'expected_fields_per_row': 31, 'field_count_errors': 0, 'field_length_errors': 0}
Data validation: {'rows': 50000, 'raw_columns': 31, 'cleaned_columns': 50, 'missing_loan_ids': 0, 'duplicate_loan_ids': 0, 'invalid_postal_codes': 0, 'invalid_borrower_groups': 0}
Cohort validation: {'rows': 50000, 'invalid_origination_years': 0, 'invalid_origination_quarters': 0, 'invalid_origination_cohorts': 0}
Saved outputs:
cleaned: e:\capstone_project\mortgage-model-reliability\data\processed\Freddie_Mac\origination_2015_clean.parquet
numeric_summary: e:\capstone_project\mortgage-model-reliability\data\processed\Freddie_Mac\origination_2015_numeric_summary.csv
categorical_summary: e:\capstone_project\mortgage-model-reliability\data\processed\Freddie_Mac\origination_2015_categorical_summary.csv
cohort_summary: e:\capstone_project\mortgage-model-reliability\data\processed\Freddie_Mac\origination_2015_cohort_summary.csv
unexpected_codes: e:\capst

### 7.1 Saved Report Verification

Read every saved quality CSV and its validation JSON. The row counts must agree
for each report type and year.

In [18]:
report_json_keys = {
    "unexpected_codes": (
        "unexpected_code_rows"
    ),
    "numeric_range_warnings": (
        "numeric_range_warning_rows"
    ),
    "definition_warnings": (
        "definition_warning_rows"
    ),
    "ratio_warnings": (
        "ratio_warning_rows"
    ),
}

saved_output_check_rows = []

for year in YEARS:
    validation_path = pipeline_results[
        year
    ]["output_paths"]["validation"]

    with validation_path.open(
        mode="r",
        encoding="utf-8",
    ) as validation_file:
        saved_validation = json.load(
            validation_file
        )

    for report_name, json_key in (
        report_json_keys.items()
    ):
        saved_report = pd.read_csv(
            pipeline_results[
                year
            ]["output_paths"][report_name]
        )

        saved_output_check_rows.append(
            {
                "year": year,
                "report": report_name,
                "saved_csv_rows": len(
                    saved_report
                ),
                "validation_json_rows": (
                    saved_validation[
                        json_key
                    ]
                ),
                "row_counts_match": (
                    len(saved_report)
                    == saved_validation[
                        json_key
                    ]
                ),
            }
        )

saved_output_check = pd.DataFrame(
    saved_output_check_rows
)

display(saved_output_check)

assert saved_output_check[
    "row_counts_match"
].all(), "Saved quality-report row counts do not match."

,year,report,saved_csv_rows,validation_json_rows,row_counts_match
0,2015,unexpected_codes,0,0,True
1,2015,numeric_range_warnings,0,0,True
2,2015,definition_warnings,0,0,True
3,2015,ratio_warnings,0,0,True
4,2016,unexpected_codes,0,0,True
5,2016,numeric_range_warnings,0,0,True
6,2016,definition_warnings,0,0,True
7,2016,ratio_warnings,0,0,True
8,2017,unexpected_codes,0,0,True
9,2017,numeric_range_warnings,0,0,True


### 7.1 Saved Cohort-Summary Verification

Each annual cohort-summary CSV is read back and compared with its in-memory
pipeline result. Its row count and total loan count are also checked against
the corresponding validation JSON. This confirms that all four quarterly
cohorts were saved without changing their counts or percentages.

In [19]:
saved_cohort_check_rows = []

for year in YEARS:
    saved_cohort_summary = pd.read_csv(
        pipeline_results[
            year
        ]["output_paths"]["cohort_summary"]
    )

    expected_cohort_summary = pipeline_results[
        year
    ]["cohort_summary"]

    try:
        pd.testing.assert_frame_equal(
            saved_cohort_summary.reset_index(
                drop=True
            ),
            expected_cohort_summary.reset_index(
                drop=True
            ),
            check_dtype=False,
            check_exact=False,
        )
        content_matches = True
    except AssertionError:
        content_matches = False

    validation_path = pipeline_results[
        year
    ]["output_paths"]["validation"]

    with validation_path.open(
        mode="r",
        encoding="utf-8",
    ) as validation_file:
        saved_validation = json.load(
            validation_file
        )

    saved_cohort_check_rows.append(
        {
            "year": year,
            "saved_rows": len(
                saved_cohort_summary
            ),
            "json_rows": saved_validation[
                "cohort_summary_rows"
            ],
            "saved_records": int(
                saved_cohort_summary["count"].sum()
            ),
            "json_records": saved_validation[
                "cohort_summary_records"
            ],
            "quarter_count": (
                saved_cohort_summary[
                    "quarter"
                ].nunique()
            ),
            "percentage_total": (
                saved_cohort_summary[
                    "percentage"
                ].sum()
            ),
            "content_matches": content_matches,
        }
    )

saved_cohort_check = pd.DataFrame(
    saved_cohort_check_rows
)

display(saved_cohort_check.round(3))

assert saved_cohort_check[
    "saved_rows"
].eq(
    saved_cohort_check["json_rows"]
).all(), (
    "Saved cohort-summary row counts do not match."
)

assert saved_cohort_check[
    "saved_records"
].eq(
    saved_cohort_check["json_records"]
).all(), (
    "Saved cohort-summary loan counts do not match."
)

assert saved_cohort_check[
    "quarter_count"
].eq(4).all(), (
    "A saved annual cohort summary is missing a quarter."
)

assert saved_cohort_check[
    "percentage_total"
].sub(100).abs().lt(1e-9).all(), (
    "Saved cohort percentages do not sum to 100%."
)

assert saved_cohort_check[
    "content_matches"
].all(), (
    "Saved cohort summaries differ from pipeline results."
)

,year,saved_rows,json_rows,saved_records,json_records,quarter_count,percentage_total,content_matches
0,2015,4,4,50000,50000,4,100.0,True
1,2016,4,4,50000,50000,4,100.0,True
2,2017,4,4,50000,50000,4,100.0,True
3,2018,4,4,50000,50000,4,100.0,True
4,2019,4,4,50000,50000,4,100.0,True
5,2020,4,4,50000,50000,4,100.0,True
6,2021,4,4,50000,50000,4,100.0,True
7,2022,4,4,50000,50000,4,100.0,True
8,2023,4,4,50000,50000,4,100.0,True


## 8. Saved-Output Verification

Read each saved Parquet dataset and run both the general data validation and the
cohort-specific validation again. This checks that serialization preserved
50,000 rows, all 31 raw fields, all 19 appended cleaned, harmonized, and cohort
fields, loan identifiers, postal-code representation, the borrower group, and
the cohort values derived from each loan identifier.

In [20]:
saved_cleaned_origination = {}
readback_rows = []

for year in YEARS:
    saved_cleaned_data = pd.read_parquet(
        pipeline_results[
            year
        ]["output_paths"]["cleaned"]
    )

    readback_validation = validate_origination(
        raw=raw_origination[year],
        cleaned=saved_cleaned_data,
        expected_rows=50000,
    )

    readback_cohort_validation = (
        validate_origination_cohorts(
            origination=saved_cleaned_data,
            expected_year=year,
        )
    )

    saved_cleaned_origination[
        year
    ] = saved_cleaned_data

    readback_rows.append(
        {
            "year": year,
            "saved_shape": (
                saved_cleaned_data.shape
            ),
            "postal_code_dtype": str(
                saved_cleaned_data[
                    "POSTAL CODE_CLEAN"
                ].dtype
            ),
            **readback_validation,
            **readback_cohort_validation,
        }
    )

readback_summary = pd.DataFrame(
    readback_rows
)

display(readback_summary)

,year,saved_shape,postal_code_dtype,rows,raw_columns,cleaned_columns,missing_loan_ids,duplicate_loan_ids,invalid_postal_codes,invalid_borrower_groups,invalid_origination_years,invalid_origination_quarters,invalid_origination_cohorts
0,2015,"(50000, 50)",string,50000,31,50,0,0,0,0,0,0,0
1,2016,"(50000, 50)",string,50000,31,50,0,0,0,0,0,0,0
2,2017,"(50000, 50)",string,50000,31,50,0,0,0,0,0,0,0
3,2018,"(50000, 50)",string,50000,31,50,0,0,0,0,0,0,0
4,2019,"(50000, 50)",string,50000,31,50,0,0,0,0,0,0,0
5,2020,"(50000, 50)",string,50000,31,50,0,0,0,0,0,0,0
6,2021,"(50000, 50)",string,50000,31,50,0,0,0,0,0,0,0
7,2022,"(50000, 50)",string,50000,31,50,0,0,0,0,0,0,0
8,2023,"(50000, 50)",string,50000,31,50,0,0,0,0,0,0,0


### Saved-Output Findings

- The pipeline saved the cleaned Parquet dataset; numeric, categorical, and
  cohort summaries; sentinel and missingness reports; all three warning reports;
  and a validation JSON for every year.
- For every warning type and year, the saved CSV row count agrees with the
  corresponding count in the validation JSON.
- Every saved cohort-summary CSV contains all four quarters, preserves the
  in-memory cohort counts and percentages, and agrees with its validation JSON.
- Every saved Parquet file reads back as 50,000 rows by 50 columns.
- Read-back validation again finds no missing or duplicate loan identifiers,
  invalid postal codes, invalid borrower groups, or invalid origination year,
  quarter, or cohort values.
- The cleaned postal-code field retains pandas string dtype after serialization.

The agreement between in-memory checks, saved report files, validation JSON,
cohort-summary CSVs, and Parquet read-back confirms that serialization preserved
the validated results.

## 9. Conclusion

The reusable Freddie Mac origination pipeline was successfully applied to all
five 2015–2019 annual samples.

### Validation outcome

- 250,000 sampled records were processed: 50,000 per year.
- Every raw row contains the expected 31 Release 47 fields.
- No field-count, maximum-length, undocumented-code, numeric-range,
  definition-timing, or ratio-rule violations were detected.
- Cleaning and cohort construction preserve the raw data and produce 50-column
  analytical datasets.
- All 20 quarterly cohorts from `2015Q1` through `2019Q4` were derived from
  valid loan identifiers, with no source-year or saved cohort-value mismatches.
- All five saved Parquet datasets and cohort-summary CSVs pass read-back
  validation, and all saved report counts agree with their validation JSON
  files.

### Cross-year interpretation

- The core credit and leverage medians are comparatively stable, but loan
  purpose, origination channel, first-time-homebuyer share, UPB, interest rate,
  and program participation change materially across years.
- The HARP-supported high-ratio segment falls steadily and disappears from the
  2019 sample, consistent with HARP's completion on December 31, 2018.
- The window captures the introduction and increasing availability of Property
  Valuation Method disclosure and the 2018 Q2 ratio and borrower-definition
  changes. The implemented cleaning, harmonization, and validation logic keeps
  these documentation changes from being misclassified as ordinary drift.
- VantageScore 4.0 is completely unavailable and should be excluded from models
  using this window.
- Property Valuation Method remains too sparsely available for a directly
  comparable five-year baseline feature. If used later, availability and
  calendar time must be handled explicitly.
- The harmonized borrower-group field should be used for comparisons spanning
  2018 Q2 instead of treating the changing raw borrower definition as stable.
- Each annual sample contains exactly 12,500 loans from each quarter. This
  balanced sample supports distribution and outcome comparisons but must not be
  interpreted as Freddie Mac's actual quarterly origination volume.

All five samples pass the current ingestion, cleaning, cohort construction,
quality, persistence, and validation requirements. The observed annual and
quarterly differences are candidates for formal drift analysis; they do not by
themselves establish statistical drift, reduced predictive performance, or
model degradation.

### Next stage

Use the verified cleaned outputs to define a leakage-safe target, measure target
event counts by cohort, and select the finest defensible evaluation cadence from
quarterly, semiannual, and annual windows. Then establish time-based development
and out-of-time evaluation cohorts and quantify feature and outcome drift with
methods appropriate to variable type and sample size. The later modeling stage
should preserve the documentation-aware exclusions and harmonizations
established here.

Documentation references:

- [Freddie Mac Single-Family Loan-Level Dataset General User Guide](https://www.freddiemac.com/fmac-resources/research/pdf/general_user_guide_july_2026.pdf)
- [Freddie Mac Release 47 disclosure changes](https://www.freddiemac.com/fmac-resources/research/pdf/disclosure-changes-summary.pdf)
- [Freddie Mac historical disclosure changes](https://capitalmarkets.freddiemac.com/crt/docs/pdfs/historical-disclosure-changes.pdf)
- [FHFA notice of HARP completion](https://www.fhfa.gov/news/news-release/fannie-mae-and-freddie-mac-refinance-volume-decreases-in-fourth-quarter-2018)